In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Import data
data = pd.read_csv('diabetes.csv')
data.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [ ]:
# Custom KNN Classifier class
class KNNClassifier:
  def __init__(self, k, distance_metric="euclidean"):
    self.k = k
    self.X_train = None
    self.y_train = None
    self.distance_metric = distance_metric

  # Saves X_train and y_train in the object
  def fit(self, X_train, y_train):
    self.X_train = X_train
    self.y_train = y_train

  # Iteratively measures distance to training data points and returns the mode of the k nearest neighbours
  def predict(self, val_data):
    X_test = val_data.iloc[:, :-1].values
    y_pred = []
    for x in X_test:
      distances = []
      for i in range(len(self.X_train)):
        distance = self.distance(x, self.X_train[i])
        distances.append((distance, self.y_train[i]))
      distances = sorted(distances, key=lambda x: x[0])
      neighbors = distances[:self.k]
      labels = [neighbor[1] for neighbor in neighbors]
      y_pred.append(max(set(labels), key=labels.count))
    accuracy = sum(y_pred == val_data.iloc[:, -1].values) / len(y_pred)
    return y_pred, accuracy

  # Distance metric
  def distance(self, x1, x2):
    if self.distance_metric == "euclidean":
      distance = 0
      for i in range(len(x1)):
        distance += (x1[i] - x2[i]) ** 2
      return distance ** 0.5
    elif self.distance_metric == "manhattan":
      distance = 0
      for i in range(len(x1)):
        distance += abs(x1[i] - x2[i])
      return distance
    else:
      raise ValueError("Invalid distance metric")


In [ ]:
# Split data into training and validation sets (80% - 20%)
split = int(0.8 * len(data))
train_data, val_data = data.iloc[: split], data.iloc[split:]
print(len(train_data), len(val_data))

614 154


In [ ]:
# Initialize KNN Classifier object and fit the training data to it
knn = KNNClassifier(k=9, distance_metric="euclidean")
knn.fit(train_data.iloc[:, :-1].values, train_data.iloc[:, -1].values)

In [ ]:
# Iterate over multiple values of k and find their accuracies
for k in range(1, 11):
  knn = KNNClassifier(k=k, distance_metric="euclidean")
  knn.fit(train_data.iloc[:, :-1].values, train_data.iloc[:, -1].values)
  y_pred, accuracy = knn.predict(val_data)
  print(f"Accuracy for k={k}: {accuracy}")

Accuracy for k=1: 0.6168831168831169
Accuracy for k=2: 0.6818181818181818
Accuracy for k=3: 0.6818181818181818
Accuracy for k=4: 0.7012987012987013
Accuracy for k=5: 0.7012987012987013
Accuracy for k=6: 0.7142857142857143
Accuracy for k=7: 0.7207792207792207
Accuracy for k=8: 0.7272727272727273
Accuracy for k=9: 0.7077922077922078
Accuracy for k=10: 0.7077922077922078


The best accuracy we got is 72.7% at k value of 8. This is a decent average, but we can improve this by preprocessing the data before sending it to our model.

In [ ]:
# Checking if our data has null values, and its distribution
print(data.info())
print(data.describe())

print(data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std    

There are no null values, but the data has 0 values for columns like Glucose, BloodPressure, SkinThickness, Insulin, BMI. None of these features can realistically have a 0 value. So we need to replace the 0 values with NaN for these columns in our data.

In [ ]:
# Replacing 0 value NaN in Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age
for i in data.columns[1:-1]:
    data[i] = data.apply(lambda x: np.nan if x[i] == 0 else x[i], axis=1)
data.isnull().sum()

,0
Pregnancies,0
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11
DiabetesPedigreeFunction,0
Age,0
Outcome,0


We can fix the missing values by either dropping the rows with null or filling them in with some appropriate value (foe example, median of the column). We will be dropping them.

In [ ]:
# Dropping rows with null values
data.dropna(inplace=True)
data.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


Outliers in the data can mislead the model into wrong predictions. We can remove them by trimming the range of each column to (Q1 - (1.5 * IQR), Q3 + (1.5 * IQR)), where IQR is the Inter Quartile Range = Q3 - Q1

In [ ]:
# Removing Outliers from the data
Q1 = data.quantile(0.25)
Q3 = data.quantile(0.75)
IQR = Q3 - Q1

data = data[~((data < (Q1 - 1.5 * IQR)) | (data > (Q3 + 1.5 * IQR))).any(axis=1)]
data.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21.0,0
6,3,78.0,50.0,32.0,88.0,31.0,0.248,26.0,1
14,5,166.0,72.0,19.0,175.0,25.8,0.587,51.0,1
16,0,118.0,84.0,47.0,230.0,45.8,0.551,31.0,1
19,1,115.0,70.0,30.0,96.0,34.6,0.529,32.0,1
20,3,126.0,88.0,41.0,235.0,39.3,0.704,27.0,0
24,11,143.0,94.0,33.0,146.0,36.6,0.254,51.0,1
25,10,125.0,70.0,26.0,115.0,31.1,0.205,41.0,1
27,1,97.0,66.0,15.0,140.0,23.2,0.487,22.0,0
31,3,158.0,76.0,36.0,245.0,31.6,0.851,28.0,1


Finally we need to appropriately scale the values of each column. We use a MinMaxScaler.

In [ ]:
# Scaling the columns
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
data = pd.DataFrame(scaler.fit_transform(data), columns=data.columns)
data.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0.090909,0.232394,0.43750,0.288462,0.241590,0.314286,0.067937,0.000000,0.0
1,0.272727,0.154930,0.18750,0.461538,0.223242,0.406349,0.135046,0.147059,1.0
2,0.454545,0.774648,0.53125,0.211538,0.489297,0.241270,0.415907,0.882353,1.0
3,0.000000,0.436620,0.71875,0.750000,0.657492,0.876190,0.386081,0.294118,1.0
4,0.090909,0.415493,0.50000,0.423077,0.247706,0.520635,0.367854,0.323529,1.0
5,0.272727,0.492958,0.78125,0.634615,0.672783,0.669841,0.512842,0.176471,0.0
6,1.000000,0.612676,0.87500,0.480769,0.400612,0.584127,0.140017,0.882353,1.0
7,0.909091,0.485915,0.50000,0.346154,0.305810,0.409524,0.099420,0.588235,1.0
8,0.090909,0.288732,0.43750,0.134615,0.382263,0.158730,0.333057,0.029412,0.0
9,0.272727,0.718310,0.59375,0.538462,0.703364,0.425397,0.634631,0.205882,1.0


In [ ]:
# Splitting the cleaned data into training and validation sets
split = int(0.8 * len(data))
train_data, val_data = data.iloc[: split], data.iloc[split:]
print(len(train_data), len(val_data))

263 66


In [ ]:
# Training KNN Classifier for different K values and different distance metrics
print("With Euclidean distance metric")
for k in range(1, 11):
  knn = KNNClassifier(k=k, distance_metric="euclidean")
  knn.fit(train_data.iloc[:, :-1].values, train_data.iloc[:, -1].values)
  y_pred, accuracy = knn.predict(val_data)
  print(f"Accuracy for k={k}: {accuracy}")

print("With Manhattan distance metric")
for k in range(1, 11):
  knn = KNNClassifier(k=k, distance_metric="manhattan")
  knn.fit(train_data.iloc[:, :-1].values, train_data.iloc[:, -1].values)
  y_pred, accuracy = knn.predict(val_data)
  print(f"Accuracy for k={k}: {accuracy}")

With Euclidean distance metric
Accuracy for k=1: 0.7272727272727273
Accuracy for k=2: 0.7727272727272727
Accuracy for k=3: 0.8484848484848485
Accuracy for k=4: 0.803030303030303
Accuracy for k=5: 0.8636363636363636
Accuracy for k=6: 0.7727272727272727
Accuracy for k=7: 0.8333333333333334
Accuracy for k=8: 0.7878787878787878
Accuracy for k=9: 0.8181818181818182
Accuracy for k=10: 0.7727272727272727
With Manhattan distance metric
Accuracy for k=1: 0.7121212121212122
Accuracy for k=2: 0.7272727272727273
Accuracy for k=3: 0.8181818181818182
Accuracy for k=4: 0.7878787878787878
Accuracy for k=5: 0.8333333333333334
Accuracy for k=6: 0.8333333333333334
Accuracy for k=7: 0.8484848484848485
Accuracy for k=8: 0.8181818181818182
Accuracy for k=9: 0.8181818181818182
Accuracy for k=10: 0.8181818181818182


The best model we got was one with euclidean distance metric with a K value of 5, with 86.3% accuracy. I believe this can still be improved if we have a bigger dataset.